In [1]:
# --- 1. Install necessary libraries ---
# 'transformers' for the LLM pipeline, 'sentencepiece' is a required dependency for the model,
# and 'tqdm' gives us a helpful progress bar.
!pip install transformers torch sentencepiece tqdm

print("✅ Libraries installed.")

✅ Libraries installed.


In [2]:
# --- 2. Load Data and Prepare for Classification ---
import pandas as pd
from tqdm.auto import tqdm

# Define the 9 harm categories that we will classify incidents into.
# These MUST be consistent.
HARM_CATEGORIES = [
    "Malicious Use & Security",
    "Fairness, Bias & Discrimination",
    "Safety, Robustness & Reliability",
    "Privacy & Data Protection",
    "Transparency & Explainability",
    "Societal & Economic Impact",
    "Human-Computer Interaction & Autonomy",
    "Data Quality & Integrity",
    "System & Task Mismatch"
]

# --- Load Data from Kaggle's input directory ---
# IMPORTANT: Replace 'your-dataset-name' with the actual folder name Kaggle created for your upload.
# You can see the folder name in the right-hand "Input" panel.
# It's usually something like 'ai-risk-hotspots-data'.
KAGGLE_INPUT_PATH = "/kaggle/input/ai-incident-data/"

df_all = pd.read_json(f"{KAGGLE_INPUT_PATH}/unified_incidents.json")
df_manual = pd.read_csv(f"{KAGGLE_INPUT_PATH}/manual_classified_incidents.csv")

# --- Identify incidents that still need classification ---
manual_ids = df_manual['incident_id'].tolist()
df_to_classify = df_all[~df_all['incident_id'].isin(manual_ids)].copy()

print(f"Total incidents loaded: {len(df_all)}")
print(f"Manually classified incidents: {len(df_manual)}")
print(f"Incidents to classify with LLM: {len(df_to_classify)}")
print("\n✅ Data loaded and prepared.")

# Display the first few rows of the data we need to classify
df_to_classify.head()

Total incidents loaded: 1238
Manually classified incidents: 200
Incidents to classify with LLM: 1038

✅ Data loaded and prepared.


,incident_id,title,description,date,entities,tags,classifications
200,203,Uber Launched Opaque Algorithm That Changes Dr...,Uber launched a new but opaque algorithm to de...,2022-02-10,[uber-drivers],[uber],"{'classifications_MIT': {'Namespace': 'MIT', '..."
201,204,A Chinese Tech Worker at Zhihu Fired Allegedly...,"The firing of an employee at Zhihu, a large Q&...",2022-02-11,"[zhihu-employees, chinese-tech-workers]",[zhihu],{'classifications_CSETv1': {'Namespace': 'CSET...
202,205,AI-Generated Profiles Used in Disinformation C...,"According to security reports by Meta, fictiti...",2022-02-25,[ukrainian-social-media-users],"[individuals-in-the-donbass-region, individual...",{'classifications_CSETv1': {'Namespace': 'CSET...
203,206,Tinder's Personalized Pricing Algorithm Found ...,Tinder’s personalized pricing was found by Con...,2015-03-01,[tinder-users-over-30-years-old],[tinder],"{'classifications_MIT': {'Namespace': 'MIT', '..."
204,207,Hawaii Police Deployed Robot Dog to Patrol a H...,Honolulu Police Department spent federal pande...,2021-01-10,[honolulu-homeless-people],[honolulu-police-department],{'classifications_CSETv1_Annotator-1': {'Names...


In [4]:
torch.cuda.is_available() 

True

In [3]:
# --- 3. Initialize the LLM Pipeline ---
from transformers import pipeline
import torch

# This command loads a pre-trained model capable of zero-shot classification.
# "zero-shot" means it can classify text into categories it has never seen during its training.
# We explicitly tell it to use the GPU (device=0) for maximum speed.
print("Loading the zero-shot classification model (facebook/bart-large-mnli)...")
print("This may take a minute or two.")

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1 # Use GPU if available
)

print("\n✅ LLM Pipeline is ready.")

2025-10-31 17:46:53.361402: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761932813.384293     169 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761932813.391139     169 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Loading the zero-shot classification model (facebook/bart-large-mnli)...
This may take a minute or two.


Device set to use cuda:0



✅ LLM Pipeline is ready.


In [6]:
# --- 4. Run LLM Classification on the Remaining Incidents ---
# We will combine the title and description for more context.
texts_to_classify = (df_to_classify['title'] + ". " + df_to_classify['description']).tolist()

# Use a batch size to make the process more efficient for the GPU
batch_size = 32
llm_classifications = []

print(f"Starting classification for {len(texts_to_classify)} incidents with a batch size of {batch_size}...")

# Use tqdm for a progress bar
for i in tqdm(range(0, len(texts_to_classify), batch_size)):
    batch = texts_to_classify[i:i + batch_size]
    
    # Get the model's predictions
    results = classifier(batch, HARM_CATEGORIES, multi_label=False)
    
    # Extract the top label for each result in the batch
    top_labels = [result['labels'][0] for result in results]
    llm_classifications.extend(top_labels)

print("\n✅ LLM classification complete.")

# Add the new classifications back to the DataFrame
df_to_classify['harm_category'] = llm_classifications
df_to_classify[['incident_id', 'title', 'harm_category']].head()

Starting classification for 1038 incidents with a batch size of 32...


  0%|          | 0/33 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



✅ LLM classification complete.


,incident_id,title,harm_category
200,203,Uber Launched Opaque Algorithm That Changes Dr...,Societal & Economic Impact
201,204,A Chinese Tech Worker at Zhihu Fired Allegedly...,Malicious Use & Security
202,205,AI-Generated Profiles Used in Disinformation C...,Malicious Use & Security
203,206,Tinder's Personalized Pricing Algorithm Found ...,"Fairness, Bias & Discrimination"
204,207,Hawaii Police Deployed Robot Dog to Patrol a H...,Societal & Economic Impact


In [9]:
# --- 5. Combine Datasets and Save Final Output ---

print("Combining manually classified data with LLM results...")

# Your df_manual DataFrame (from Cell 2) contains the first 200 classified incidents.
# Your df_to_classify DataFrame (from Cell 4) now contains the remaining incidents with the new 'harm_category' column.

# Before we combine, let's make sure both dataframes have the same columns in the same order.
# The 'df_manual' dataframe already has the correct structure.
# We'll select the same columns from 'df_to_classify'.
final_columns = ['incident_id', 'date', 'title', 'description', 'harm_category']
df_llm_classified = df_to_classify[final_columns]

# Now, concatenate the two dataframes together into one final master dataframe.
df_final = pd.concat([df_manual, df_llm_classified], ignore_index=True)

# For good measure, let's sort the entire dataset by incident_id.
df_final.sort_values('incident_id', inplace=True)


# --- Verification Step ---
print("\n--- Verification ---")
print(f"Total incidents in the final, combined dataset: {len(df_final)}")
print("Showing the first 5 rows (from your manual/programmatic classification):")
print(df_final.head())
print("\nShowing the last 5 rows (from the LLM classification):")
print(df_final.tail())


# --- Save the final result to a CSV file ---
output_filename = 'classified_incidents.csv'
df_final.to_csv(output_filename, index=False)

print(f"\n✅ Success! The complete dataset has been saved to '{output_filename}'.")
print("You can find this file in the 'Output' section on the right side of your Kaggle notebook.")

Combining manually classified data with LLM results...

--- Verification ---
Total incidents in the final, combined dataset: 1238
Showing the first 5 rows (from your manual/programmatic classification):
   incident_id        date                                              title  \
0            1  2015-05-19  Google’s YouTube Kids App Presents Inappropria...   
1            2  2018-12-05  Warehouse robot ruptures can of bear spray and...   
2            3  2018-10-27  Crashes with Maneuvering Characteristics Augme...   
3            4  2018-03-18               Uber AV Killed Pedestrian in Arizona   
4            5  2015-07-13         Collection of Robotic Surgery Malfunctions   

                                         description  \
0  YouTube’s content filtering and recommendation...   
1  Twenty-four Amazon workers in New Jersey were ...   
2  A Boeing 737 crashed into the sea, killing 189...   
3  An Uber autonomous vehicle (AV) in autonomous ...   
4  Study on database reports o